In [1]:
import sys
print(sys.executable)

/home/osbdet/notebooks/Group Project IOT/venv/bin/python


In [2]:
import pandas as pd
# ---- Kafka ----
KAFKA_BOOTSTRAP = "localhost:9092"
KAFKA_TOPIC = "sensor_topic"

# ---- MariaDB ----
MARIADB_HOST = "localhost"
MARIADB_PORT = 3306
MARIADB_DB   = "iot"
MARIADB_USER = "iotuser"
MARIADB_PASS = "iotpass123"

# Table we will write into
MARIADB_TABLE = "sensor_events"
STAGING_TABLE = f"{MARIADB_TABLE}_staging"

# Spark checkpoint directory (must be writable)
CHECKPOINT_DIR = "/tmp/spark_checkpoints/sensor_to_mariadb"

print("Kafka:", KAFKA_BOOTSTRAP, KAFKA_TOPIC)
print("MariaDB:", MARIADB_HOST, MARIADB_DB, MARIADB_TABLE)

Kafka: localhost:9092 sensor_topic
MariaDB: localhost iot sensor_events


In [3]:
import pymysql

conn = pymysql.connect(
    host=MARIADB_HOST,
    port=MARIADB_PORT,
    user=MARIADB_USER,
    password=MARIADB_PASS,
    autocommit=True
)
cur = conn.cursor()

cur.execute(f"CREATE DATABASE IF NOT EXISTS {MARIADB_DB};")
cur.execute(f"USE {MARIADB_DB};")

# Drop & recreate fresh (dev mode). Comment these out if you want to keep existing data.
cur.execute(f"DROP TABLE IF EXISTS {STAGING_TABLE};")
cur.execute(f"DROP TABLE IF EXISTS {MARIADB_TABLE};")

cur.execute(f"""
CREATE TABLE {MARIADB_TABLE} (
    device_id VARCHAR(100) NOT NULL,
    event_time_ms BIGINT NOT NULL,
    event_time DATETIME(3) NOT NULL,
    temp_c DOUBLE NOT NULL,
    vib INT NOT NULL,

    kafka_topic VARCHAR(255),
    kafka_partition INT,
    kafka_offset BIGINT,

    ingest_time DATETIME(3) DEFAULT CURRENT_TIMESTAMP(3),

    PRIMARY KEY (device_id, event_time_ms),
    INDEX idx_event_time (event_time),
    INDEX idx_device_time (device_id, event_time)
);
""")

# staging table with same schema
cur.execute(f"CREATE TABLE {STAGING_TABLE} LIKE {MARIADB_TABLE};")

cur.close()
conn.close()
print("✅ DB + tables ready")

0

0

0

0

0

0

✅ DB + tables ready


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("KafkaToMariaDBUpsert")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,"
        "mysql:mysql-connector-java:8.0.33"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark started:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/20 19:47:26 WARN Utils: Your hostname, osbdet, resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
26/03/20 19:47:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/osbdet/notebooks/Group%20Project%20IOT/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/osbdet/.ivy2.5.2/cache
The jars for the packages stored in: /home/osbdet/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
mysql#mysql-connector-java added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-01ca3a67-5760-4a8b-b5b8-52a7cd7f1a2d;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in cen

✅ Spark started: 4.1.1


In [5]:
raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .selectExpr("CAST(value AS STRING) AS v")
)

debug_q = raw.writeStream.format("console").option("truncate", False).start()

26/03/20 19:48:14 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-f6d3cc7a-b31a-4912-925c-1154eefcb0ac. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/03/20 19:48:15 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [6]:
from pyspark.sql.functions import col, from_json, expr
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, IntegerType

schema = StructType([
    StructField("device_id", StringType(), True),
    StructField("event_time_ms", LongType(), True),
    StructField("temp_c", DoubleType(), True),
    StructField("vib", IntegerType(), True),
])

kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")  # use "earliest" if you want to backfill
    .load()
)

parsed = (
    kafka_df
    .select(
        col("topic").cast("string").alias("kafka_topic"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
        col("value").cast("string").alias("value_str")
    )
    .withColumn("json", from_json(col("value_str"), schema))
    .select(
        col("json.device_id").alias("device_id"),
        col("json.event_time_ms").alias("event_time_ms"),
        expr("CAST(from_unixtime(json.event_time_ms/1000.0) AS TIMESTAMP)").alias("event_time"),
        col("json.temp_c").alias("temp_c"),
        col("json.vib").alias("vib"),
        col("kafka_topic"),
        col("kafka_partition"),
        col("kafka_offset"),
    )
    .where(
        col("device_id").isNotNull() &
        col("event_time_ms").isNotNull() &
        col("temp_c").isNotNull() &
        col("vib").isNotNull()
    )
)

# Optional: preview stream in console for sanity (temporary)
# display(parsed)  # Databricks-style
print("✅ Kafka stream parsed")

✅ Kafka stream parsed


-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------------------------------------------------------------------------------+
|v                                                                                           |
+--------------------------------------------------------------------------------------------+
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774032320430, "temp_c": 34.05, "vib": 31}|
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774032321439, "temp_c": 34.1, "vib": 25} |
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774032322442, "temp_c": 35.21, "vib": 34}|
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774032323452, "temp_c": 35.73, "vib": 34}|
+--------------------------------------------------------------------------------------------+



In [7]:
import pymysql

JDBC_URL = (
    f"jdbc:mysql://{MARIADB_HOST}:{MARIADB_PORT}/{MARIADB_DB}"
    f"?useSSL=false&rewriteBatchedStatements=true&serverTimezone=UTC"
)

JDBC_PROPS = {
    "user": MARIADB_USER,
    "password": MARIADB_PASS,
    "driver": "com.mysql.cj.jdbc.Driver",
}

def upsert_batch(batch_df, batch_id: int):
    if batch_df.rdd.isEmpty():
        return

    # 1) Append batch to staging
    (batch_df.write
        .mode("append")
        .jdbc(JDBC_URL, STAGING_TABLE, properties=JDBC_PROPS)
    )

    # 2) Upsert staging -> target, then truncate staging
    conn = pymysql.connect(
        host=MARIADB_HOST,
        port=MARIADB_PORT,
        user=MARIADB_USER,
        password=MARIADB_PASS,
        database=MARIADB_DB,
        autocommit=True
    )
    try:
        with conn.cursor() as cur:
            cur.execute(f"""
                INSERT INTO {MARIADB_TABLE}
                (device_id, event_time_ms, event_time, temp_c, vib, kafka_topic, kafka_partition, kafka_offset)
                SELECT device_id, event_time_ms, event_time, temp_c, vib, kafka_topic, kafka_partition, kafka_offset
                FROM {STAGING_TABLE}
                ON DUPLICATE KEY UPDATE
                    event_time = VALUES(event_time),
                    temp_c = VALUES(temp_c),
                    vib = VALUES(vib),
                    kafka_topic = VALUES(kafka_topic),
                    kafka_partition = VALUES(kafka_partition),
                    kafka_offset = VALUES(kafka_offset);
            """)
            cur.execute(f"TRUNCATE TABLE {STAGING_TABLE};")
    finally:
        conn.close()

query = (
    parsed.writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", CHECKPOINT_DIR)
    .outputMode("update")   # required by Spark; foreachBatch controls actual output
    .start()
)

print("✅ Streaming started")

26/03/20 19:48:43 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


✅ Streaming started


-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------------------------------------------------------------------------------+
|v                                                                                          |
+-------------------------------------------------------------------------------------------+
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774032324730, "temp_c": 36.0, "vib": 29}|
+-------------------------------------------------------------------------------------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+--------------------------------------------------------------------------------------------+
|v                                                                                           |
+--------------------------------------------------------------------------------------------+
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774032325739, "temp_c": 35.47, "vib": 31}|
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774032326742, "temp_c": 36.6, "vib": 23} |
+--------------------------------------------------------------------------------------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+--------------------------------------------------------------------------------------------+
|v                                                                                           |
+--------------------------------------------------------------------------------------------+
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774032327752, "temp_c": 36.14, "vib": 27}|
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774032328755, "temp_c": 36.4, "vib": 24} |
+--------------------------------------------------------------------------------------------+

-------------------------------------------
Batch: 4
-------------------------------------------
+--------------------------------------------------------------------------------------------+
|v                                                                                           |
+--------------------------------------------

...

-------------------------------------------
Batch: 278
-------------------------------------------
+--------------------------------------------------------------------------------------------+
|v                                                                                           |
+--------------------------------------------------------------------------------------------+
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774033136002, "temp_c": 31.35, "vib": 15}|
+--------------------------------------------------------------------------------------------+



-------------------------------------------
Batch: 279
-------------------------------------------
+--------------------------------------------------------------------------------------------+
|v                                                                                           |
+--------------------------------------------------------------------------------------------+
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774033137011, "temp_c": 31.02, "vib": 10}|
+--------------------------------------------------------------------------------------------+



-------------------------------------------
Batch: 280
-------------------------------------------
+--------------------------------------------------------------------------------------------+
|v                                                                                           |
+--------------------------------------------------------------------------------------------+
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774033138015, "temp_c": 31.51, "vib": 15}|
+--------------------------------------------------------------------------------------------+



-------------------------------------------
Batch: 281
-------------------------------------------
+--------------------------------------------------------------------------------------------+
|v                                                                                           |
+--------------------------------------------------------------------------------------------+
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774033139264, "temp_c": 31.78, "vib": 16}|
+--------------------------------------------------------------------------------------------+



-------------------------------------------
Batch: 282
-------------------------------------------
+--------------------------------------------------------------------------------------------+
|v                                                                                           |
+--------------------------------------------------------------------------------------------+
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1774033140268, "temp_c": 32.27, "vib": 40}|
+--------------------------------------------------------------------------------------------+



In [19]:
import pymysql, pandas as pd

conn = pymysql.connect(
    host=MARIADB_HOST,
    port=MARIADB_PORT,
    user=MARIADB_USER,
    password=MARIADB_PASS,
    database=MARIADB_DB
)

df = pd.read_sql(f"SELECT * FROM {MARIADB_TABLE} ORDER BY event_time_ms DESC LIMIT 20;", conn)
conn.close()

df

/tmp/ipykernel_4923/408603171.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {MARIADB_TABLE} ORDER BY event_time_ms DESC LIMIT 20;", conn)


,device_id,event_time_ms,event_time,temp_c,vib,kafka_topic,kafka_partition,kafka_offset,ingest_time
0,esp32-wokwi-001,1772515996235,2026-03-03 05:33:16,46.52,30,sensor_topic,0,56,2026-03-03 06:38:26.072
1,esp32-wokwi-001,1772515995226,2026-03-03 05:33:15,45.59,28,sensor_topic,0,55,2026-03-03 06:38:26.072
2,esp32-wokwi-001,1772515994222,2026-03-03 05:33:14,45.57,31,sensor_topic,0,54,2026-03-03 06:38:23.426
3,esp32-wokwi-001,1772515991115,2026-03-03 05:33:11,45.42,53,sensor_topic,0,53,2026-03-03 06:35:29.202
4,esp32-wokwi-001,1772515990106,2026-03-03 05:33:10,45.18,25,sensor_topic,0,52,2026-03-03 06:35:29.202
5,esp32-wokwi-001,1772515989102,2026-03-03 05:33:09,44.11,30,sensor_topic,0,51,2026-03-03 06:35:29.202
6,esp32-wokwi-001,1772515988094,2026-03-03 05:33:08,44.22,32,sensor_topic,0,50,2026-03-03 06:35:29.202
7,esp32-wokwi-001,1772515987090,2026-03-03 05:33:07,43.58,33,sensor_topic,0,49,2026-03-03 06:35:29.202
8,esp32-wokwi-001,1772515986081,2026-03-03 05:33:06,43.30,29,sensor_topic,0,48,2026-03-03 06:35:29.202
9,esp32-wokwi-001,1772515985077,2026-03-03 05:33:05,42.52,27,sensor_topic,0,47,2026-03-03 06:35:29.202


In [10]:
import pymysql

conn = pymysql.connect(
    host=MARIADB_HOST,
    port=MARIADB_PORT,
    user=MARIADB_USER,
    password=MARIADB_PASS,
    database=MARIADB_DB
)

with conn.cursor() as cur:
    cur.execute(f"SELECT COUNT(*) FROM {MARIADB_TABLE};")
    row_count = cur.fetchone()[0]

conn.close()

print("Total rows in table:", row_count)

1

Total rows in table: 562


-------------------------------------------
Batch: 306
-------------------------------------------
+--------------------------------------------------------------------------------------------+
|v                                                                                           |
+--------------------------------------------------------------------------------------------+
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1772897616316, "temp_c": 68.86, "vib": 27}|
+--------------------------------------------------------------------------------------------+



-------------------------------------------
Batch: 307
-------------------------------------------
+--------------------------------------------------------------------------------------------+
|v                                                                                           |
+--------------------------------------------------------------------------------------------+
|{"device_id": "esp32-wokwi-001", "event_time_ms": 1772897617324, "temp_c": 67.97, "vib": 31}|
+--------------------------------------------------------------------------------------------+



In [11]:
import pymysql
import pandas as pd

conn = pymysql.connect(
    host=MARIADB_HOST,
    port=MARIADB_PORT,
    user=MARIADB_USER,
    password=MARIADB_PASS,
    database=MARIADB_DB
)

query = f"SELECT * FROM {MARIADB_TABLE};"

df = pd.read_sql(query, conn)

conn.close()

df.to_csv("output.csv", index=False)

print("CSV file saved as output.csv")

CSV file saved as output.csv


/tmp/ipykernel_2135/2123085448.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [12]:
query.stop()
print("🛑 Streaming stopped")

AttributeError: 'str' object has no attribute 'stop'

In [13]:
# Stop streaming queries if any exist
try:
    for q in spark.streams.active:
        q.stop()
except Exception:
    pass

# Stop Spark session
try:
    spark.stop()
except Exception:
    pass

print("🧹 Stopped Spark + streams (if they existed).")

26/03/07 16:50:58 WARN DAGScheduler: Failed to cancel job group 03ee6598-8ec4-4968-b15a-5b0634955ca0. Cannot find active jobs for it.
26/03/07 16:50:58 WARN DAGScheduler: Failed to cancel job group 03ee6598-8ec4-4968-b15a-5b0634955ca0. Cannot find active jobs for it.
26/03/07 16:50:58 WARN DAGScheduler: Failed to cancel job group c30f143d-a9fd-43ac-ad11-c9fb0db94f4b. Cannot find active jobs for it.
26/03/07 16:50:58 WARN DAGScheduler: Failed to cancel job group c30f143d-a9fd-43ac-ad11-c9fb0db94f4b. Cannot find active jobs for it.


🧹 Stopped Spark + streams (if they existed).


In [14]:
import shutil, os

if os.path.exists(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)
    print("✅ Deleted checkpoint:", CHECKPOINT_DIR)
else:
    print("Checkpoint dir not found, OK.")

✅ Deleted checkpoint: /tmp/spark_checkpoints/sensor_to_mariadb


In [ ]:
#To clean MariaDB careful

import pymysql

conn = pymysql.connect(
    host=MARIADB_HOST,
    port=MARIADB_PORT,
    user=MARIADB_USER,
    password=MARIADB_PASS,
    database=MARIADB_DB,
    autocommit=True
)

with conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE sensor_events;")

conn.close()
print("✅ MariaDB table cleaned")